# Modul 02: ML-Grundbegriffe, Grenzen und Lernarten

    **Notebooktyp:** Übungen mit ausführlichen Lösungen  
    **Vorlesungen dieses Moduls:** Begriffe und Grenzen, Lernarten erkennen  
    **Erwarteter Schwierigkeitsgrad:** Grundlagen mit ersten scikit-learn-Anwendungen  
    **Orientierungszeit:** etwa 75 bis 100 Minuten

    ## Überblick

    Sie unterscheiden regelbasierte Automatisierung von datenbasiertem Lernen und ordnen kleine Problemstellungen den wichtigsten Lernarten zu. Mehrere bewusst kleine Modelle dienen dazu, Begriffe, Nutzen, Grenzen und Risiken praktisch zu prüfen.

    ## Verwendete Vorlesungsnotebooks

    Die Aufgaben wurden aus dem Inhalt beider Vorlesungen dieses Moduls abgeleitet:

    - `ML Für Anfänger - Record_Module_02A_20260723.ipynb`
- `ML Für Anfänger - Record_Module_02B_20260723.ipynb`

    ## Colab-Kompatibilität

    Dieses Notebook ist für die kostenlose Version von Google Colab ausgelegt. Die Daten sind eingebaut, synthetisch erzeugt oder öffentlich verfügbar. Modelle und Trainingsbudgets sind bewusst klein gehalten. Führen Sie die Zellen in der vorgegebenen Reihenfolge aus.

## Lernziele

    Nach der Bearbeitung sollen Sie:

    - Feste Programmregeln von datenbasiert gelernten Mustern unterscheiden.
- Zentrale Begriffe wie Daten, Merkmale, Zielwert, Training, Modell und Inferenz erklären.
- Einfache Szenarien hinsichtlich Nutzen und Grenzen von ML bewerten.
- Überwachtes und unüberwachtes Lernen unterscheiden.
- Klassifikation, Regression, Clustering und Anomalieerkennung zuordnen.
- Die Wahl einer Lernart für einfache Alltagsprobleme begründen.

    ## Bewertete Fähigkeiten

    - regelbasierte Funktionen und einfache ML-Modelle gegenüberstellen
- Problem, Eingaben, Ziel und Erfolgskriterium formulieren
- Klassifikation, Regression, Clustering und Anomalieerkennung praktisch erkennen
- Risiken, Grenzen, Rückmeldung und menschliche Kontrolle dokumentieren

## Arbeitsanweisungen

Dieses Lösungsnotebook entspricht dem Übungsnotebook Aufgabe für Aufgabe. Führen Sie es von oben nach unten aus und vergleichen Sie nicht nur Endwerte, sondern auch Vorgehen, Formprüfungen, Datenaufteilung und Interpretation. Die Kommentare erklären bewusst auch typische Fehlerquellen und methodische Entscheidungen.

- Führen Sie zuerst das gemeinsame Setup aus.
- Verändern Sie vorgegebene Splits und Seeds nur, wenn eine Aufgabe dies ausdrücklich erlaubt.
- Prüfen Sie Formen, Datentypen und Wertebereiche frühzeitig.
- Begründen Sie Modell-, Metrik- und Visualisierungsentscheidungen.
- Achten Sie auf Datenleckage und eine saubere Trennung von Training, Validierung und Test.

## Gemeinsames Setup

Führen Sie diese Zelle einmal aus, bevor Sie mit Aufgabe 1 beginnen.

In [ ]:
# Gemeinsames Setup für dieses Notebook
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.datasets import load_iris, make_blobs
from sklearn.ensemble import IsolationForest
from sklearn.linear_model import LinearRegression
from sklearn.metrics import accuracy_score, mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)
np.set_printoptions(precision=3, suppress=True)
pd.set_option("display.max_columns", 50)
warnings.filterwarnings("ignore", category=FutureWarning)

# Ein kleiner Datensatz für eine regelbasierte und eine datenbasierte Entscheidung.
support_cases = pd.DataFrame(
    {
        "waiting_minutes": [2, 5, 8, 12, 18, 25, 30, 4, 15, 22, 7, 27],
        "message_length": [30, 80, 120, 150, 220, 260, 310, 60, 180, 240, 100, 290],
        "contains_refund_word": [0, 0, 1, 0, 1, 1, 1, 0, 0, 1, 0, 1],
        "urgent": [0, 0, 1, 0, 1, 1, 1, 0, 0, 1, 0, 1],
    }
)
print(support_cases.head())

print("Setup abgeschlossen. Zufallsstartwert:", RANDOM_SEED)


## Aufgabe 1: Feste Regeln implementieren und einordnen

    Schreiben Sie eine Funktion `rule_based_priority`, die einen Supportfall als `"hoch"` einstuft, wenn mindestens eine der folgenden Regeln erfüllt ist:

- Wartezeit mindestens 20 Minuten, oder
- das Wort für Rückerstattung ist enthalten und die Nachricht hat mindestens 200 Zeichen.

Andernfalls soll `"normal"` zurückgegeben werden. Wenden Sie die Funktion zeilenweise auf `support_cases` an und vergleichen Sie das Ergebnis mit dem vorhandenen Ziel `urgent`.

> **Hinweis:** Fragen Sie sich, ob sich das Verhalten ohne Trainingsdaten vollständig erklären lässt.

In [ ]:
# ============================================================


In [ ]:
# ============================================================
# KOMMENTIERTE MUSTERLÖSUNG: Feste Regeln implementieren und einordnen
#
# Ziel dieser Codezelle:
# Schreiben Sie eine Funktion rulebasedpriority, die einen Supportfall als "hoch"
# einstuft, wenn mindestens eine der folgenden Regeln erfüllt ist: - Wartezeit
# mindestens 20 Minuten, oder - das Wort für Rückerstattung is...
#
# Die Lösung folgt bewusst einer gut prüfbaren Schrittfolge.
# Zwischenwerte und Ausgaben machen Formen, Annahmen und Ergebnisse sichtbar.
# Die fachliche Interpretation und typische Fehlerquellen stehen in der
# ausführlichen Markdown-Reflexion direkt unter dieser Codezelle.
# ============================================================

def rule_based_priority(row: pd.Series) -> str:
    # Ordnet einen Supportfall mit vollständig festgelegten Regeln ein.

    # Regel 1 verwendet nur die Wartezeit.
    long_wait = row["waiting_minutes"] >= 20

    # Regel 2 kombiniert zwei Bedingungen. Beide müssen gleichzeitig gelten.
    refund_and_long_message = (
        row["contains_refund_word"] == 1
        and row["message_length"] >= 200
    )

    # Da die Regeln vorab von Menschen formuliert wurden, findet hier
    # kein Training und kein Lernen aus Daten statt.
    return "hoch" if (long_wait or refund_and_long_message) else "normal"

rule_results = support_cases.copy()
rule_results["rule_priority"] = rule_results.apply(
    rule_based_priority, axis=1
)

# Für einen einfachen Vergleich wird die Textausgabe in 0 und 1 übersetzt.
rule_results["rule_prediction"] = (
    rule_results["rule_priority"] == "hoch"
).astype(int)

rule_accuracy = accuracy_score(
    rule_results["urgent"], rule_results["rule_prediction"]
)

print(rule_results.to_string(index=False))
print("Regelgenauigkeit:", round(rule_accuracy, 3))

### Reflexion zu Aufgabe 1

Die Funktion ist regelbasiert, weil jede Entscheidung vollständig aus vorher festgelegten Bedingungen folgt. Es gibt weder eine Trainingsphase noch gelernte Parameter. Regeln sind gut geeignet, wenn die Bedingungen stabil, nachvollziehbar und vollständig formulierbar sind. Sie werden problematisch, wenn viele Ausnahmen auftreten oder sich Muster häufig ändern.

**Kontrollfrage:** Welche Annahme, Formprüfung oder Trennungsentscheidung war für die Korrektheit dieser Lösung besonders wichtig?

## Aufgabe 2: Überwachtes Lernen: Klassifikation und Regression

    Bearbeiten Sie zwei kleine überwachte Lernaufgaben.

**Klassifikation:** Laden Sie Iris, teilen Sie die Daten stratifiziert auf, trainieren Sie einen kleinen Entscheidungsbaum und berechnen Sie die Genauigkeit.

**Regression:** Erzeugen Sie aus `study_hours = [1, 2, ..., 10]` und passenden Prüfungspunkten einen Trainings- und Testsatz, trainieren Sie eine lineare Regression und berechnen Sie den mittleren absoluten Fehler.

Geben Sie für beide Aufgaben Merkmale, Zielwert und Art der Vorhersage an.

> **Hinweis:** Die Art des Zielwerts bestimmt, ob eine überwachte Aufgabe Klassifikation oder Regression ist.

In [ ]:
# ============================================================


In [ ]:
# ============================================================
# KOMMENTIERTE MUSTERLÖSUNG: Überwachtes Lernen: Klassifikation und Regression
#
# Ziel dieser Codezelle:
# Bearbeiten Sie zwei kleine überwachte Lernaufgaben. Klassifikation: Laden Sie
# Iris, teilen Sie die Daten stratifiziert auf, trainieren Sie einen kleinen
# Entscheidungsbaum und berechnen Sie die Genauigkeit. Regression:...
#
# Die Lösung folgt bewusst einer gut prüfbaren Schrittfolge.
# Zwischenwerte und Ausgaben machen Formen, Annahmen und Ergebnisse sichtbar.
# Die fachliche Interpretation und typische Fehlerquellen stehen in der
# ausführlichen Markdown-Reflexion direkt unter dieser Codezelle.
# ============================================================

# -----------------------------
# Teil A: Klassifikation
# -----------------------------
iris = load_iris()
X_iris = iris.data
y_iris = iris.target

# stratify=y_iris hält die Klassenanteile in Train und Test ähnlich.
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_iris,
    y_iris,
    test_size=0.30,
    random_state=RANDOM_SEED,
    stratify=y_iris,
)

# Ein flacher Baum bleibt für diesen Einführungsschritt nachvollziehbar.
classifier = DecisionTreeClassifier(
    max_depth=3,
    random_state=RANDOM_SEED,
)
classifier.fit(X_train_c, y_train_c)
class_predictions = classifier.predict(X_test_c)
classification_accuracy = accuracy_score(y_test_c, class_predictions)

# -----------------------------
# Teil B: Regression
# -----------------------------
study_hours = np.arange(1, 11, dtype=float).reshape(-1, 1)
exam_points = np.array([43, 47, 52, 56, 61, 66, 70, 76, 81, 85], dtype=float)

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    study_hours,
    exam_points,
    test_size=0.30,
    random_state=RANDOM_SEED,
)

regressor = LinearRegression()
regressor.fit(X_train_r, y_train_r)
value_predictions = regressor.predict(X_test_r)
regression_mae = mean_absolute_error(y_test_r, value_predictions)

print("Klassifikation: Genauigkeit =", round(classification_accuracy, 3))
print("Regression: MAE =", round(regression_mae, 3))
print("Regressionsvorhersagen:", np.round(value_predictions, 1))

### Reflexion zu Aufgabe 2

Bei Iris sind die vier gemessenen Blütenmerkmale die Eingaben und die Pflanzenart ist ein kategorialer Zielwert. Daher handelt es sich um Klassifikation. Bei den Prüfungspunkten ist die Lernzeit das Merkmal und die Punktzahl ein numerischer Zielwert. Daher handelt es sich um Regression. Beide Aufgaben sind überwacht, weil zu den Trainingsbeobachtungen bekannte Zielwerte vorliegen.

**Kontrollfrage:** Welche Annahme, Formprüfung oder Trennungsentscheidung war für die Korrektheit dieser Lösung besonders wichtig?

## Aufgabe 3: Unüberwachtes Lernen: Gruppen und ungewöhnliche Muster

    Erzeugen Sie mit `make_blobs` einen zweidimensionalen Datensatz mit drei Gruppen. Trainieren Sie `KMeans` mit drei Clustern und visualisieren Sie Punkte und Zentren. Ergänzen Sie anschließend drei weit entfernte Punkte und verwenden Sie `IsolationForest`, um ungewöhnliche Beobachtungen zu markieren.

Vergleichen Sie Clusterlabels und Anomalielabels mit echten fachlichen Zielwerten: Was fehlt beiden Verfahren?

> **Hinweis:** Ein vom Algorithmus erzeugtes Label ist nicht automatisch eine fachlich bestätigte Klasse.

In [ ]:
# ============================================================


In [ ]:
# ============================================================
# KOMMENTIERTE MUSTERLÖSUNG: Unüberwachtes Lernen: Gruppen und ungewöhnliche Muster
#
# Ziel dieser Codezelle:
# Erzeugen Sie mit makeblobs einen zweidimensionalen Datensatz mit drei Gruppen.
# Trainieren Sie KMeans mit drei Clustern und visualisieren Sie Punkte und Zentren.
# Ergänzen Sie anschließend drei weit entfernte Punkte und...
#
# Die Lösung folgt bewusst einer gut prüfbaren Schrittfolge.
# Zwischenwerte und Ausgaben machen Formen, Annahmen und Ergebnisse sichtbar.
# Die fachliche Interpretation und typische Fehlerquellen stehen in der
# ausführlichen Markdown-Reflexion direkt unter dieser Codezelle.
# ============================================================

# make_blobs erzeugt nur für die spätere Kontrolle bekannte Gruppen.
# KMeans erhält diese Labels beim Training jedoch nicht.
X_blobs, hidden_group_labels = make_blobs(
    n_samples=150,
    centers=3,
    cluster_std=0.75,
    random_state=RANDOM_SEED,
)

kmeans = KMeans(
    n_clusters=3,
    n_init=10,
    random_state=RANDOM_SEED,
)
cluster_labels = kmeans.fit_predict(X_blobs)

fig, ax = plt.subplots(figsize=(7, 5))
scatter = ax.scatter(
    X_blobs[:, 0], X_blobs[:, 1], c=cluster_labels, alpha=0.75
)
ax.scatter(
    kmeans.cluster_centers_[:, 0],
    kmeans.cluster_centers_[:, 1],
    marker="X",
    s=180,
    label="Zentren",
)
ax.set_title("Von KMeans gefundene Gruppen")
ax.set_xlabel("Merkmal 1")
ax.set_ylabel("Merkmal 2")
ax.legend()
plt.tight_layout()
plt.show()

# Drei bewusst weit entfernte Punkte simulieren ungewöhnliche Fälle.
unusual_points = np.array([[10, 10], [-10, 8], [8, -9]], dtype=float)
X_with_outliers = np.vstack([X_blobs, unusual_points])

# contamination gibt den erwarteten ungefähren Anteil ungewöhnlicher
# Beobachtungen an. -1 bedeutet Anomalie, +1 bedeutet normal.
detector = IsolationForest(
    contamination=3 / len(X_with_outliers),
    random_state=RANDOM_SEED,
)
anomaly_labels = detector.fit_predict(X_with_outliers)
detected_anomalies = X_with_outliers[anomaly_labels == -1]

print("Als ungewöhnlich markierte Punkte:")
print(np.round(detected_anomalies, 2))

### Reflexion zu Aufgabe 3

KMeans erzeugt Gruppen nach geometrischer Ähnlichkeit, aber die Clusterzahlen `0`, `1` und `2` besitzen zunächst keine fachliche Bedeutung. IsolationForest markiert statistisch ungewöhnliche Punkte, weiß jedoch nicht, ob diese Fehler, seltene legitime Fälle oder wichtige Ereignisse sind. Beide Verfahren arbeiten ohne bekannte Zielwerte. Fachliche Benennung und Bewertung müssen deshalb nachträglich erfolgen.

**Kontrollfrage:** Welche Annahme, Formprüfung oder Trennungsentscheidung war für die Korrektheit dieser Lösung besonders wichtig?

## Aufgabe 4: Problemstellungen passend zuordnen

    Erstellen Sie eine Tabelle mit den folgenden Szenarien und ordnen Sie jeweils eine geeignete Vorgehensweise zu: `feste Regeln`, `Klassifikation`, `Regression`, `Clustering`, `Anomalieerkennung`, `Reinforcement Learning` oder `keine Automatisierung`.

- Spam-E-Mail erkennen
- zukünftigen Energieverbrauch als Zahl schätzen
- Kundengruppen ohne vorhandene Labels entdecken
- seltene Maschinenschwingungen markieren
- Aufzugsteuerung durch Belohnung verbessern
- gesetzlich exakt definierte Altersgrenze prüfen
- einmalige ethische Entscheidung ohne ausreichende Daten automatisieren

Ergänzen Sie zu jeder Zeile eine kurze Begründung sowie benötigte Daten oder Rückmeldung.

> **Hinweis:** Formulieren Sie zuerst die gewünschte Ausgabe und fragen Sie danach, welche Rückmeldung verfügbar ist.

In [ ]:
scenarios = [
    "Spam-E-Mail erkennen",
    "Energieverbrauch schätzen",
    "Kundengruppen entdecken",
    "Seltene Maschinenschwingungen markieren",
    "Aufzugsteuerung verbessern",
    "Gesetzliche Altersgrenze prüfen",
    "Einmalige ethische Entscheidung automatisieren",
]

# ============================================================


In [ ]:
# ============================================================
# KOMMENTIERTE MUSTERLÖSUNG: Problemstellungen passend zuordnen
#
# Ziel dieser Codezelle:
# Erstellen Sie eine Tabelle mit den folgenden Szenarien und ordnen Sie jeweils eine
# geeignete Vorgehensweise zu: feste Regeln, Klassifikation, Regression, Clustering,
# Anomalieerkennung, Reinforcement Learning oder kein...
#
# Die Lösung folgt bewusst einer gut prüfbaren Schrittfolge.
# Zwischenwerte und Ausgaben machen Formen, Annahmen und Ergebnisse sichtbar.
# Die fachliche Interpretation und typische Fehlerquellen stehen in der
# ausführlichen Markdown-Reflexion direkt unter dieser Codezelle.
# ============================================================

scenarios = [
    "Spam-E-Mail erkennen",
    "Energieverbrauch schätzen",
    "Kundengruppen entdecken",
    "Seltene Maschinenschwingungen markieren",
    "Aufzugsteuerung verbessern",
    "Gesetzliche Altersgrenze prüfen",
    "Einmalige ethische Entscheidung automatisieren",
]

# Die Zuordnung wird bewusst als nachvollziehbare Tabelle und nicht als
# bloße Liste erstellt. So lassen sich Annahmen und Datenbedarf prüfen.
task_mapping = pd.DataFrame(
    {
        "scenario": scenarios,
        "approach": [
            "Klassifikation",
            "Regression",
            "Clustering",
            "Anomalieerkennung",
            "Reinforcement Learning",
            "feste Regeln",
            "keine Automatisierung",
        ],
        "needed_information": [
            "E-Mails mit bestätigtem Spam-/Nicht-Spam-Label",
            "historischer Verbrauch und erklärende Merkmale",
            "Kundenmerkmale, aber keine Gruppenlabels",
            "viele normale Signale und möglichst geprüfte Ereignisse",
            "Zustände, Aktionen und eine geeignete Belohnung",
            "Geburtsdatum, Stichtag und eindeutige Rechtsregel",
            "ausreichende Daten, legitimes Ziel und verantwortliche Kontrolle fehlen",
        ],
        "reason": [
            "bekannte diskrete Zielklasse",
            "kontinuierlicher numerischer Zielwert",
            "Struktur soll ohne Zielwerte entdeckt werden",
            "seltene Abweichungen von typischem Verhalten",
            "Folgen von Aktionen werden über Belohnung bewertet",
            "Entscheidung ist vollständig und stabil formulierbar",
            "Automatisierung wäre nicht belastbar oder verantwortbar",
        ],
    }
)

print(task_mapping.to_string(index=False))

### Reflexion zu Aufgabe 4

Die passende Lernart ergibt sich nicht allein aus dem verwendeten Datentyp. Entscheidend sind Ziel, verfügbare Rückmeldung und gewünschte Ausgabe. Ein Problem sollte nicht mit ML gelöst werden, wenn eine klare Regel genügt, wenn kein legitimes Erfolgskriterium existiert oder wenn die Folgen ohne menschliche Verantwortung nicht vertretbar sind.

**Kontrollfrage:** Welche Annahme, Formprüfung oder Trennungsentscheidung war für die Korrektheit dieser Lösung besonders wichtig?

## Aufgabe 5: Integrationsaufgabe: Regel oder lernendes Modell?

    Vergleichen Sie für `support_cases` die Regel aus Aufgabe 1 mit einem kleinen Entscheidungsbaum.

1. Teilen Sie Merkmale und Zielwert in Train und Test auf.
2. Trainieren Sie einen Baum mit `max_depth=2`.
3. Berechnen Sie die Testgenauigkeit.
4. Wenden Sie die feste Regel auf dieselben Testzeilen an und berechnen Sie ebenfalls die Genauigkeit.
5. Entscheiden Sie, welches Verfahren Sie für einen echten Pilotbetrieb bevorzugen würden. Berücksichtigen Sie Datenmenge, Nachvollziehbarkeit, mögliche Änderungen und menschliche Kontrolle.

> **Hinweis:** Eine höhere Testzahl auf wenigen Fällen ist noch kein ausreichender Nachweis für den produktiven Einsatz.

In [ ]:
features = ["waiting_minutes", "message_length", "contains_refund_word"]
X = support_cases[features]
y = support_cases["urgent"]

# ============================================================


In [ ]:
# ============================================================
# KOMMENTIERTE MUSTERLÖSUNG: Integrationsaufgabe: Regel oder lernendes Modell?
#
# Ziel dieser Codezelle:
# Vergleichen Sie für supportcases die Regel aus Aufgabe 1 mit einem kleinen
# Entscheidungsbaum. 1. Teilen Sie Merkmale und Zielwert in Train und Test auf. 2.
# Trainieren Sie einen Baum mit maxdepth=2. 3. Berechnen Sie di...
#
# Die Lösung folgt bewusst einer gut prüfbaren Schrittfolge.
# Zwischenwerte und Ausgaben machen Formen, Annahmen und Ergebnisse sichtbar.
# Die fachliche Interpretation und typische Fehlerquellen stehen in der
# ausführlichen Markdown-Reflexion direkt unter dieser Codezelle.
# ============================================================

features = ["waiting_minutes", "message_length", "contains_refund_word"]
X = support_cases[features]
y = support_cases["urgent"]

# Ein fester Split ermöglicht einen direkten Vergleich auf denselben Fällen.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.33,
    random_state=RANDOM_SEED,
    stratify=y,
)

# Der kleine Baum begrenzt die Komplexität und bleibt leichter erklärbar.
tree_model = DecisionTreeClassifier(
    max_depth=2,
    random_state=RANDOM_SEED,
)
tree_model.fit(X_train, y_train)
tree_predictions = tree_model.predict(X_test)
tree_accuracy = accuracy_score(y_test, tree_predictions)

# Die Regel wird exakt auf dieselben Testzeilen angewendet. Dadurch ist
# der Vergleich nicht durch unterschiedliche Testfälle verzerrt.
def rule_prediction_for_features(row: pd.Series) -> int:
    long_wait = row["waiting_minutes"] >= 20
    refund_and_long_message = (
        row["contains_refund_word"] == 1
        and row["message_length"] >= 200
    )
    return int(long_wait or refund_and_long_message)

rule_predictions = X_test.apply(rule_prediction_for_features, axis=1)
rule_accuracy = accuracy_score(y_test, rule_predictions)

comparison = pd.DataFrame(
    {
        "actual": y_test,
        "tree_prediction": tree_predictions,
        "rule_prediction": rule_predictions,
    },
    index=y_test.index,
).sort_index()

print(comparison)
print("Baumgenauigkeit:", round(tree_accuracy, 3))
print("Regelgenauigkeit:", round(rule_accuracy, 3))

### Reflexion zu Aufgabe 5

Wegen der sehr kleinen Stichprobe ist eine einzelne Testgenauigkeit unsicher. Ein Pilot könnte zunächst die transparente Regel verwenden und parallel Daten sammeln. Das lernende Modell wäre interessant, wenn genügend repräsentative Fälle, stabile Labels und ein Überwachungsprozess vorhanden sind. Unabhängig vom Verfahren sollten Fehlentscheidungen überprüfbar sein und dringende Fälle nicht ohne menschliche Eskalationsmöglichkeit verloren gehen.

**Kontrollfrage:** Welche Annahme, Formprüfung oder Trennungsentscheidung war für die Korrektheit dieser Lösung besonders wichtig?

## Abschluss und Selbstkontrolle

Prüfen Sie nach dem Durcharbeiten, ob Sie jede Lösung ohne bloßes Kopieren erklären könnten. Achten Sie besonders auf die Stellen, an denen Datenleckage, unpassende Formen, falsche Metriken oder unkontrollierte Zufälligkeit zu scheinbar guten, aber methodisch falschen Ergebnissen führen könnten.

- Alle Aufgaben und Unterpunkte wurden bearbeitet.
- Verwendete Seeds und Datenpartitionen sind nachvollziehbar.
- Testdaten wurden nicht vorzeitig für Entscheidungen genutzt.
- Ergebnisse werden vorsichtig und fachlich begründet interpretiert.
- Es gibt keine hardcodierten lokalen Dateipfade oder privaten Zugangsdaten.